# GLIF3 Single Neuron Simulation

Explore the dynamics of a Generalized Leaky Integrate-and-Fire (GLIF3) neuron.

## Table of Contents
1. [GLIF3 Model Overview](#1-glif3-model-overview)
2. [State Variables and Parameters](#2-state-variables-and-parameters)
3. [Creating a Single Neuron](#3-creating-a-single-neuron)
4. [Step Current Response](#4-step-current-response)
5. [Refractory Period and Adaptation](#5-refractory-period-and-adaptation)
6. [PSC Dynamics](#6-psc-dynamics)
7. [Spike Raster Plot](#7-spike-raster-plot)

---

In [ ]:
# Setup
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

# Enable high-DPI display for plots
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

## 1. GLIF3 Model Overview

The **GLIF3** (Generalized Leaky Integrate-and-Fire Level 3) neuron model extends the standard LIF model with **adaptive spike currents (ASC)**.

### Standard LIF Dynamics

$$
\tau_m \frac{dV}{dt} = -(V - E_L) + R \cdot I_{syn}
$$

where:
- $V$ is the membrane potential
- $E_L$ is the leak reversal potential
- $\tau_m = C_m / g$ is the membrane time constant
- $I_{syn}$ is the synaptic input current

### GLIF3 Additions

GLIF3 adds two **adaptive spike currents** that provide spike-rate adaptation:

$$
\frac{dI_{asc,i}}{dt} = -k_i \cdot I_{asc,i} + A_i \cdot \delta(t - t_{spike})
$$

These currents:
- Jump by amplitude $A_i$ after each spike
- Decay exponentially with rate $k_i$
- Typically hyperpolarizing (negative), reducing firing rate

### Model Hierarchy

| Model | Features |
|-------|----------|
| LIF | Leaky membrane, threshold, reset |
| GLIF1 | LIF + refractory period |
| GLIF2 | GLIF1 + reset voltage dynamics |
| **GLIF3** | GLIF2 + **adaptive spike currents** |
| GLIF4 | GLIF3 + threshold dynamics |
| GLIF5 | GLIF4 + subthreshold currents |

## 2. State Variables and Parameters

The GLIF3 model maintains several state variables that evolve over time.

In [ ]:
from v1_jax.nn.glif3_cell import GLIF3State, GLIF3Params

# Display state variables
print("GLIF3State fields:")
print("="*60)
state_docs = {
    'z_buf': 'Spike history buffer for synaptic delays (batch, n_neurons * max_delay)',
    'v': 'Membrane potential - normalized (batch, n_neurons)',
    'r': 'Refractory counter in ms (batch, n_neurons)',
    'asc_1': 'Adaptive spike current channel 1 (batch, n_neurons)',
    'asc_2': 'Adaptive spike current channel 2 (batch, n_neurons)',
    'psc_rise': 'PSC rising phase (batch, n_neurons * n_receptors)',
    'psc': 'Post-synaptic current (batch, n_neurons * n_receptors)',
}

for field in GLIF3State._fields:
    print(f"  {field:12s}: {state_docs[field]}")

In [ ]:
# Display parameter descriptions
print("\nGLIF3Params fields:")
print("="*60)
param_docs = {
    'v_reset': 'Reset potential after spike (normalized)',
    'v_th': 'Firing threshold (normalized)',
    'e_l': 'Leak reversal potential (normalized)',
    't_ref': 'Absolute refractory period (ms)',
    'decay': 'Membrane decay factor: exp(-dt/tau_m)',
    'current_factor': 'Current to voltage conversion factor',
    'syn_decay': 'Synaptic decay per receptor (n_neurons, n_receptors)',
    'psc_initial': 'PSC initial amplitude per receptor',
    'param_k': 'ASC decay rates for 2 channels (n_neurons, 2)',
    'asc_amps': 'ASC amplitudes for 2 channels (n_neurons, 2)',
    'param_g': 'Membrane conductance',
    'voltage_scale': 'Scale for denormalization (V_th - E_L)',
    'voltage_offset': 'Offset for denormalization (E_L)',
}

for field in GLIF3Params._fields:
    print(f"  {field:15s}: {param_docs[field]}")

### Voltage Normalization

The V1 model uses **normalized voltages** for numerical stability:

$$
V_{norm} = \frac{V - E_L}{V_{th} - E_L}
$$

In normalized coordinates:
- $E_L \rightarrow 0$ (leak potential becomes 0)
- $V_{th} \rightarrow 1$ (threshold becomes 1)

To recover physical voltages:
$$
V = V_{norm} \cdot (V_{th} - E_L) + E_L = V_{norm} \cdot scale + offset
$$

## 3. Creating a Single Neuron

Let's create a single GLIF3 neuron with biologically realistic parameters.

In [ ]:
from v1_jax.nn.glif3_cell import GLIF3Cell, GLIF3Params, GLIF3State, glif3_step

# Neuron configuration
n_neurons = 1
n_receptors = 4  # AMPA, NMDA, GABA_A, GABA_B
max_delay = 5
batch_size = 1
dt = 1.0  # ms

# Biologically inspired parameters
# Typical cortical pyramidal neuron values
V_th_phys = -50.0    # mV - threshold
E_L_phys = -70.0     # mV - leak potential
V_reset_phys = -70.0 # mV - reset potential (same as leak)

# Normalization
voltage_scale = V_th_phys - E_L_phys  # 20 mV
voltage_offset = E_L_phys

# Normalized values
v_th_norm = (V_th_phys - voltage_offset) / voltage_scale  # = 1.0
e_l_norm = (E_L_phys - voltage_offset) / voltage_scale    # = 0.0
v_reset_norm = (V_reset_phys - voltage_offset) / voltage_scale  # = 0.0

print(f"Physical voltages: V_th={V_th_phys}mV, E_L={E_L_phys}mV, V_reset={V_reset_phys}mV")
print(f"Normalized:        V_th={v_th_norm:.1f}, E_L={e_l_norm:.1f}, V_reset={v_reset_norm:.1f}")
print(f"Scale={voltage_scale:.1f}mV, Offset={voltage_offset:.1f}mV")

In [ ]:
# Create GLIF3 parameters
# Time constant tau_m = C_m / g, typical ~20ms for cortical neurons
tau_m = 20.0  # ms
decay = jnp.exp(-dt / tau_m)
current_factor = (1 / tau_m) * (1 - jnp.exp(-dt / tau_m)) * tau_m

# Synaptic time constants (4 receptors)
tau_syn = jnp.array([2.0, 100.0, 6.0, 150.0])  # AMPA, NMDA, GABA_A, GABA_B
syn_decay = jnp.exp(-dt / tau_syn)
psc_initial = jnp.e / tau_syn

# Adaptive spike current parameters
# Two channels with different time constants
k_asc = jnp.array([0.05, 0.01])  # Decay rates (1/ms)
asc_amps = jnp.array([-0.5, -0.3])  # Amplitudes (negative = hyperpolarizing)

params = GLIF3Params(
    v_reset=jnp.array([v_reset_norm]),
    v_th=jnp.array([v_th_norm]),
    e_l=jnp.array([e_l_norm]),
    t_ref=jnp.array([2.0]),              # 2ms absolute refractory period
    decay=jnp.array([decay]),
    current_factor=jnp.array([current_factor]),
    syn_decay=syn_decay.reshape(1, 4),
    psc_initial=psc_initial.reshape(1, 4),
    param_k=k_asc.reshape(1, 2),
    asc_amps=asc_amps.reshape(1, 2),
    param_g=jnp.array([1.0 / tau_m]),
    voltage_scale=jnp.array([voltage_scale]),
    voltage_offset=jnp.array([voltage_offset]),
)

print("Parameters created successfully!")
print(f"Membrane time constant: {tau_m} ms")
print(f"Membrane decay per step: {decay:.4f}")
print(f"Refractory period: {float(params.t_ref[0])} ms")
print(f"ASC amplitudes: {asc_amps} (normalized)")

In [ ]:
# Initialize state
state = GLIF3Cell.init_state(
    n_neurons=n_neurons,
    n_receptors=n_receptors,
    max_delay=max_delay,
    batch_size=batch_size,
    params=params,
)

print("Initial state:")
print(f"  v (membrane potential): {state.v}")
print(f"  r (refractory counter): {state.r}")
print(f"  asc_1: {state.asc_1}")
print(f"  asc_2: {state.asc_2}")
print(f"  z_buf shape: {state.z_buf.shape}")
print(f"  psc shape: {state.psc.shape}")

## 4. Step Current Response

Let's simulate the neuron's response to a constant input current.

In [ ]:
# Simulation parameters
sim_time = 200  # ms
input_amplitude = 3.0  # Input current strength

# Surrogate gradient parameters
gauss_std = 0.5
dampening_factor = 0.3

# Storage for results
voltages = []
spikes = []
asc1_trace = []
asc2_trace = []
refractory_trace = []

# Reset state
state = GLIF3Cell.init_state(
    n_neurons=n_neurons,
    n_receptors=n_receptors,
    max_delay=max_delay,
    batch_size=batch_size,
    params=params,
)

# Simulation loop
for t in range(sim_time):
    # Constant input current (applied to all receptors)
    # Divide by n_receptors so total input is input_amplitude
    inputs = jnp.ones((batch_size, n_neurons * n_receptors)) * (input_amplitude / n_receptors)
    rec_current = jnp.zeros_like(inputs)  # No recurrent input
    
    state, z, v = glif3_step(
        params=params,
        state=state,
        inputs=inputs,
        recurrent_current=rec_current,
        n_neurons=n_neurons,
        n_receptors=n_receptors,
        max_delay=max_delay,
        dt=dt,
        gauss_std=gauss_std,
        dampening_factor=dampening_factor,
    )
    
    voltages.append(float(v[0, 0]))
    spikes.append(float(z[0, 0]))
    asc1_trace.append(float(state.asc_1[0, 0]))
    asc2_trace.append(float(state.asc_2[0, 0]))
    refractory_trace.append(float(state.r[0, 0]))

# Convert to arrays
voltages = np.array(voltages)
spikes = np.array(spikes)
asc1_trace = np.array(asc1_trace)
asc2_trace = np.array(asc2_trace)
refractory_trace = np.array(refractory_trace)

spike_times = np.where(spikes > 0.5)[0]
print(f"Simulation complete: {len(spike_times)} spikes in {sim_time}ms")
print(f"Mean firing rate: {len(spike_times) / sim_time * 1000:.1f} Hz")

In [ ]:
# Plot results
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
time = np.arange(sim_time)

# 1. Membrane potential
axes[0].plot(time, voltages, 'b-', linewidth=1)
axes[0].scatter(spike_times, [voltages[t] for t in spike_times],
                c='red', s=50, zorder=5, label='Spikes')
axes[0].axhline(y=V_th_phys, color='gray', linestyle='--', alpha=0.7, label=f'Threshold ({V_th_phys}mV)')
axes[0].axhline(y=E_L_phys, color='green', linestyle=':', alpha=0.7, label=f'Leak ({E_L_phys}mV)')
axes[0].set_ylabel('Voltage (mV)')
axes[0].set_title(f'GLIF3 Neuron Response to Step Current (I={input_amplitude})')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# 2. Adaptive spike currents
axes[1].plot(time, asc1_trace, 'orange', linewidth=1.5, label='ASC channel 1 (fast)')
axes[1].plot(time, asc2_trace, 'purple', linewidth=1.5, label='ASC channel 2 (slow)')
axes[1].plot(time, asc1_trace + asc2_trace, 'k--', linewidth=1, alpha=0.7, label='Total ASC')
axes[1].set_ylabel('ASC (normalized)')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# 3. Refractory state
axes[2].fill_between(time, 0, refractory_trace, alpha=0.5, color='red')
axes[2].set_ylabel('Refractory (ms)')
axes[2].set_ylim([0, params.t_ref[0] + 0.5])
axes[2].grid(True, alpha=0.3)

# 4. Spike raster
axes[3].eventplot([spike_times], lineoffsets=0, linelengths=0.8, colors='black')
axes[3].set_ylabel('Spikes')
axes[3].set_xlabel('Time (ms)')
axes[3].set_yticks([])
axes[3].set_xlim([0, sim_time])

plt.tight_layout()
plt.show()

## 5. Refractory Period and Adaptation

Let's examine how the **refractory period** and **adaptive spike currents** affect firing patterns.

In [ ]:
def simulate_neuron(params, sim_time=200, input_amp=3.0):
    """Run simulation and return spike times."""
    state = GLIF3Cell.init_state(
        n_neurons=1, n_receptors=4, max_delay=5, batch_size=1, params=params
    )
    
    spike_times = []
    for t in range(sim_time):
        inputs = jnp.ones((1, 4)) * (input_amp / 4)
        state, z, v = glif3_step(
            params, state, inputs, jnp.zeros_like(inputs),
            1, 4, 5, 1.0, 0.5, 0.3
        )
        if float(z[0, 0]) > 0.5:
            spike_times.append(t)
    
    return np.array(spike_times)

# Test different refractory periods
t_refs = [0.5, 2.0, 5.0, 10.0]
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for idx, t_ref in enumerate(t_refs):
    # Create params with different refractory period
    test_params = params._replace(t_ref=jnp.array([t_ref]))
    spike_times = simulate_neuron(test_params, sim_time=200, input_amp=3.0)
    
    # Compute ISI (inter-spike intervals)
    if len(spike_times) > 1:
        isi = np.diff(spike_times)
        mean_isi = np.mean(isi)
        rate = 1000 / mean_isi if mean_isi > 0 else 0
    else:
        isi = []
        rate = len(spike_times) * 5  # spikes per 200ms -> Hz
    
    axes[idx].eventplot([spike_times], lineoffsets=0, linelengths=0.8, colors='black')
    axes[idx].set_xlim([0, 200])
    axes[idx].set_yticks([])
    axes[idx].set_xlabel('Time (ms)')
    axes[idx].set_title(f't_ref = {t_ref}ms | Rate: {rate:.1f} Hz | Spikes: {len(spike_times)}')

plt.suptitle('Effect of Refractory Period on Firing', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Test different ASC amplitudes (adaptation strength)
asc_configs = [
    (0.0, 0.0, 'No adaptation'),
    (-0.2, -0.1, 'Weak adaptation'),
    (-0.5, -0.3, 'Medium adaptation'),
    (-1.0, -0.6, 'Strong adaptation'),
]

fig, axes = plt.subplots(len(asc_configs), 1, figsize=(14, 8), sharex=True)

for idx, (a1, a2, label) in enumerate(asc_configs):
    test_params = params._replace(
        asc_amps=jnp.array([[a1, a2]])
    )
    spike_times = simulate_neuron(test_params, sim_time=300, input_amp=3.0)
    
    # Compute instantaneous firing rate
    if len(spike_times) > 1:
        isi = np.diff(spike_times)
        initial_rate = 1000 / isi[0] if isi[0] > 0 else 0
        final_rate = 1000 / isi[-1] if len(isi) > 0 and isi[-1] > 0 else 0
    else:
        initial_rate = final_rate = 0
    
    axes[idx].eventplot([spike_times], lineoffsets=0, linelengths=0.8, colors='black')
    axes[idx].set_xlim([0, 300])
    axes[idx].set_yticks([])
    axes[idx].set_ylabel(label, fontsize=10)
    axes[idx].set_title(f'ASC=({a1}, {a2}) | Initial: {initial_rate:.0f}Hz, Final: {final_rate:.0f}Hz', 
                        fontsize=10, loc='right')

axes[-1].set_xlabel('Time (ms)')
plt.suptitle('Effect of Adaptive Spike Currents on Firing Rate Adaptation', fontsize=14)
plt.tight_layout()
plt.show()

print("Note: Negative ASC amplitudes cause hyperpolarization after spikes,")
print("reducing subsequent firing rate (spike-frequency adaptation).")

## 6. PSC Dynamics

The GLIF3 model uses **4 receptor types** with different time constants:

| Receptor | Type | $\tau_{syn}$ (ms) | Function |
|----------|------|-------------------|----------|
| AMPA | Excitatory | ~2 | Fast excitation |
| NMDA | Excitatory | ~100 | Slow excitation |
| GABA_A | Inhibitory | ~6 | Fast inhibition |
| GABA_B | Inhibitory | ~150 | Slow inhibition |

The PSC uses **dual-exponential dynamics**:
$$
\text{psc\_rise}[t+1] = \alpha \cdot \text{psc\_rise}[t] + \text{input} \cdot \text{psc\_initial}
$$
$$
\text{psc}[t+1] = \alpha \cdot \text{psc}[t] + dt \cdot \alpha \cdot \text{psc\_rise}[t]
$$

where $\alpha = e^{-dt/\tau_{syn}}$

In [ ]:
from v1_jax.nn.synaptic import psc_dynamics, DEFAULT_TAU_SYN

# Visualize PSC response to a single spike
sim_time = 300
dt = 1.0

# Initialize PSC state
psc_rise = jnp.zeros((1, 4))  # (batch, receptors)
psc = jnp.zeros((1, 4))

# Synaptic parameters
tau_syn = DEFAULT_TAU_SYN  # [2, 100, 6, 150]
syn_decay = jnp.exp(-dt / tau_syn)
psc_initial = jnp.e / tau_syn

receptor_names = ['AMPA', 'NMDA', 'GABA_A', 'GABA_B']
psc_traces = {name: [] for name in receptor_names}

for t in range(sim_time):
    # Single spike at t=10
    spike = jnp.array([[1.0, 1.0, 1.0, 1.0]]) if t == 10 else jnp.zeros((1, 4))
    
    psc_rise, psc = psc_dynamics(
        spikes=spike,
        psc_rise=psc_rise,
        psc=psc,
        syn_decay=syn_decay,
        psc_initial=psc_initial,
        dt=dt
    )
    
    for i, name in enumerate(receptor_names):
        psc_traces[name].append(float(psc[0, i]))

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
colors = ['red', 'darkred', 'blue', 'darkblue']
time = np.arange(sim_time)

for idx, (name, color) in enumerate(zip(receptor_names, colors)):
    trace = np.array(psc_traces[name])
    axes[idx].plot(time, trace, color=color, linewidth=2)
    axes[idx].fill_between(time, 0, trace, alpha=0.3, color=color)
    axes[idx].axvline(x=10, color='gray', linestyle='--', alpha=0.5, label='Spike')
    axes[idx].set_xlabel('Time (ms)')
    axes[idx].set_ylabel('PSC')
    axes[idx].set_title(f'{name} ($\\tau$ = {float(tau_syn[idx]):.0f}ms)')
    axes[idx].grid(True, alpha=0.3)
    
    # Mark peak
    peak_idx = np.argmax(trace)
    axes[idx].scatter([peak_idx], [trace[peak_idx]], c='black', s=50, zorder=5)
    axes[idx].annotate(f'peak @ {peak_idx}ms', (peak_idx, trace[peak_idx]),
                       xytext=(peak_idx+20, trace[peak_idx]*0.9))

plt.suptitle('Post-Synaptic Current (PSC) Dynamics for Each Receptor Type', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Compare all receptors on same plot
plt.figure(figsize=(12, 5))

for name, color in zip(receptor_names, colors):
    trace = np.array(psc_traces[name])
    # Normalize for comparison
    trace_norm = trace / np.max(trace)
    tau = float(tau_syn[receptor_names.index(name)])
    plt.plot(time, trace_norm, color=color, linewidth=2, 
             label=f'{name} ($\\tau$={tau:.0f}ms)')

plt.axvline(x=10, color='gray', linestyle='--', alpha=0.5, label='Spike')
plt.xlabel('Time (ms)')
plt.ylabel('Normalized PSC')
plt.title('Comparison of Receptor Time Constants')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim([0, 200])
plt.show()

print("Key observations:")
print("- AMPA: Fast rise and decay, mediates quick excitation")
print("- NMDA: Slow dynamics, maintains prolonged depolarization")
print("- GABA_A: Fast inhibition, shapes spike timing")
print("- GABA_B: Slow inhibition, provides sustained suppression")

## 7. Spike Raster Plot

Let's simulate multiple neurons with different input currents and visualize the population activity.

In [ ]:
# Simulate multiple neurons (different input strengths)
n_test_neurons = 20
sim_time = 500
input_range = jnp.linspace(1.5, 5.0, n_test_neurons)

all_spike_times = []

for neuron_idx in range(n_test_neurons):
    input_amp = float(input_range[neuron_idx])
    
    # Use original params
    state = GLIF3Cell.init_state(
        n_neurons=1, n_receptors=4, max_delay=5, batch_size=1, params=params
    )
    
    spike_times = []
    for t in range(sim_time):
        inputs = jnp.ones((1, 4)) * (input_amp / 4)
        state, z, v = glif3_step(
            params, state, inputs, jnp.zeros_like(inputs),
            1, 4, 5, 1.0, 0.5, 0.3
        )
        if float(z[0, 0]) > 0.5:
            spike_times.append(t)
    
    all_spike_times.append(spike_times)

# Plot raster
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Raster plot
for neuron_idx, spike_times in enumerate(all_spike_times):
    if len(spike_times) > 0:
        axes[0].scatter(spike_times, [neuron_idx]*len(spike_times), 
                       c='black', s=2, marker='|')

axes[0].set_xlabel('Time (ms)')
axes[0].set_ylabel('Neuron (by input strength)')
axes[0].set_title('Spike Raster Plot')
axes[0].set_xlim([0, sim_time])
axes[0].set_ylim([-0.5, n_test_neurons - 0.5])

# Add colorbar-like annotation for input strength
ax_cb = axes[0].twinx()
ax_cb.set_ylabel('Input Strength', color='blue')
ax_cb.set_ylim([float(input_range[0]), float(input_range[-1])])
ax_cb.tick_params(axis='y', labelcolor='blue')

# F-I curve (firing rate vs input)
firing_rates = [len(spikes) / sim_time * 1000 for spikes in all_spike_times]

axes[1].plot(input_range, firing_rates, 'o-', color='blue', linewidth=2, markersize=8)
axes[1].set_xlabel('Input Current')
axes[1].set_ylabel('Firing Rate (Hz)')
axes[1].set_title('F-I Curve (Frequency vs Input)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Input range: {float(input_range[0]):.1f} to {float(input_range[-1]):.1f}")
print(f"Firing rate range: {min(firing_rates):.1f} to {max(firing_rates):.1f} Hz")

In [ ]:
# Population firing rate histogram over time
bin_size = 10  # ms
n_bins = sim_time // bin_size

# Count spikes in each bin
population_rate = np.zeros(n_bins)
for spike_times in all_spike_times:
    for t in spike_times:
        bin_idx = t // bin_size
        if bin_idx < n_bins:
            population_rate[bin_idx] += 1

# Convert to rate (spikes per neuron per second)
population_rate = population_rate / n_test_neurons / (bin_size / 1000)

plt.figure(figsize=(12, 4))
time_bins = np.arange(n_bins) * bin_size + bin_size/2
plt.bar(time_bins, population_rate, width=bin_size*0.9, color='steelblue', alpha=0.8)
plt.axhline(y=np.mean(population_rate), color='red', linestyle='--', 
            label=f'Mean: {np.mean(population_rate):.1f} Hz')
plt.xlabel('Time (ms)')
plt.ylabel('Population Rate (Hz)')
plt.title('Population Firing Rate Over Time')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.show()

---

## Summary

Key takeaways:

1. **GLIF3 Model**: Extends LIF with adaptive spike currents (ASC) that provide spike-frequency adaptation

2. **State Variables**:
   - `v`: Membrane potential (normalized)
   - `r`: Refractory counter
   - `asc_1, asc_2`: Two adaptive current channels with different time constants
   - `psc`: Post-synaptic current (4 receptor types)
   - `z_buf`: Spike buffer for synaptic delays

3. **Voltage Normalization**: 
   - Improves numerical stability
   - $V_{norm} = (V - E_L) / (V_{th} - E_L)$

4. **PSC Dynamics**:
   - 4 receptor types: AMPA, NMDA, GABA_A, GABA_B
   - Dual-exponential model for realistic synaptic currents

5. **Adaptation**:
   - ASC channels provide negative feedback after spikes
   - Results in decreased firing rate over sustained input

---

## Exercises

1. **Vary refractory period**: Change `t_ref` from 1ms to 10ms and observe the maximum possible firing rate.

2. **Explore ASC dynamics**: Modify `param_k` (ASC decay rates) and observe how it affects adaptation time course.

3. **Input-Output relationship**: Create an F-I curve by sweeping input current and measuring steady-state firing rate.

In [ ]:
# Exercise starter: Explore ASC decay rates

# Try different k values
k_configs = [
    # (k1, k2, label)
    (0.1, 0.05, 'Fast decay'),
    (0.05, 0.01, 'Medium decay (default)'),
    (0.02, 0.005, 'Slow decay'),
]

print("TODO: Implement ASC decay rate comparison")
print("Hint: Modify param_k in GLIF3Params and compare adaptation patterns")

---

## Source Files

- `src/v1_jax/nn/glif3_cell.py` - GLIF3 neuron implementation
- `src/v1_jax/nn/synaptic.py` - PSC dynamics and synaptic filters
- `src/v1_jax/nn/spike_functions.py` - Surrogate gradient spike functions

## References

- Chen et al., "Data-driven models of visual cortex", Science Advances 2022
- Allen Institute GLIF Models: https://allensdk.readthedocs.io/en/latest/glif_models.html
- Teeter et al., "Generalized leaky integrate-and-fire models classify multiple neuron types", Nature Communications 2018